# CALION - District Heating Optimization

Komplettes Notebook für Optimierung, Analyse und Visualisierung.

## Inhalt
1. **Setup** - Imports und Konfiguration
2. **Optimierung** - Workflow ausführen
3. **Ergebnisse** - KPIs und Zusammenfassung
4. **Netzwerk** - Thermische Netzwerk-Analyse
5. **Visualisierung** - Plots und Dashboard

---

## 1. Setup

In [9]:
# Bootstrap
import sys
from pathlib import Path

current = Path.cwd()
for candidate in [current] + list(current.parents):
    if (candidate / 'calion').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        PROJECT_ROOT = candidate
        break

from calion.io.notebook_helpers import setup_notebook_environment
PROJECT_ROOT = setup_notebook_environment()

print(f"\n✅ Projekt: {PROJECT_ROOT}")

✅ Projekt-Root: c:\Users\LKR\Documents\GitHub\Energy_Framwork\Planing-Framework-for-Heat
✅ Matplotlib konfiguriert (Standard backend)
✅ Pandas konfiguriert
✅ Warnings unterdrückt

✅ Projekt: c:\Users\LKR\Documents\GitHub\Energy_Framwork\Planing-Framework-for-Heat


In [10]:
# Imports
import pandas as pd
import numpy as np
from datetime import datetime

from calion.run import rolling_horizon as rh
from calion.config.merge import load_and_merge
from calion.io.notebook_helpers import (
    save_workflow_run,
    display_workflow_summary,
    display_kpi_summary,
    list_saved_workflows,
    load_workflow_from_saved
)

print("✅ Imports erfolgreich")

✅ Imports erfolgreich


## 2. Konfiguration

### Modulare Konfigurationsstruktur

```
configs/
├── 00_base/          # Grundeinstellungen (Solver, Grid, Kosten)
├── 01_tech/          # Technische Parameter (Brennstoffe, Generatoren, WP)
├── 02_site/          # Standort-Daten (Stadtbach)
├── 03_systems/       # System-Konfigurationen (technical_limits, defaults, costs)
├── 04_scenarios/     # Szenarien mit optimization-Sektion
├── 05_networks/      # Thermische Netzwerke
├── presets/          # Vorkonfigurierte Kombinationen
└── _deprecated/      # Legacy-Konfigurationen
```

### Neue Optimization-Konfiguration (04_scenarios)

Die Optimierung wird jetzt explizit in den Szenarien konfiguriert:

```yaml
optimization:
  variables:                    # Was soll optimiert werden?
    heat_pumps:
      capacity: true            # HP-Kapazitäten optimieren
      build: true               # HP-Bau-Entscheidungen optimieren
    storage:
      capacity: true            # Speicher-Kapazität optimieren
      power: true               # Speicher-Leistung optimieren
      build: true               # Speicher-Bau-Entscheidung optimieren
  fix_after_window: 0           # Nach Window 0 Design fixieren (für RH)
```

### Rolling Horizon Terminal Policies

| Policy | Beschreibung | Feasibility |
|--------|--------------|-------------|
| `equal` | SOC_end == SOC_init | Kann scheitern |
| `soft` | Weiche Nebenbedingung | **Immer lösbar** |
| `value` | Diminishing Returns Value Function | **Immer lösbar** |
| `free` | Keine Constraint | Immer lösbar |

**Aktuell empfohlen:** `terminal_policy: value` mit diminishing returns

In [ ]:
# ============================================================
# KONFIGURATION - Hier anpassen!
# ============================================================

# EMPFOHLEN: Preset verwenden (vorkonfigurierte Kombinationen)
# ============================================================

# Option 1: Rolling Horizon mit Value Function (EMPFOHLEN für RH-Test)
CONFIG_PATHS = ['configs/presets/rh_full_system.yaml']

# Option 2: Schneller Test (1 Woche, Baseline)
# CONFIG_PATHS = ['configs/presets/quick_test.yaml']

# Option 3: Wärmepumpen-Optimierung (Full Year)
# CONFIG_PATHS = ['configs/presets/hp_optimization.yaml']

# Option 4: Speicher-Studie (Full Year)
# CONFIG_PATHS = ['configs/presets/storage_study.yaml']

# Option 5: Legacy Config (alte monolithische Config)
# CONFIG_PATHS = ['configs/stadtbach.yaml']

# ALTERNATIVE: Modulare Konfiguration (für Experten)
# ============================================================
# CONFIG_PATHS = [
#     'configs/00_base/solver.yaml',
#     'configs/00_base/grid.yaml', 
#     'configs/00_base/costs.yaml',
#     'configs/01_tech/fuels.yaml',
#     'configs/01_tech/generators.yaml',
#     'configs/01_tech/heat_pumps.yaml',
#     'configs/01_tech/storage.yaml',
#     'configs/02_site/stadtbach/data_source.yaml',
#     'configs/02_site/stadtbach/assets.yaml',
#     'configs/03_systems/full.yaml',          # System mit technical_limits, defaults, costs
#     'configs/04_scenarios/rh_q1_2023.yaml',  # Szenario mit optimization-Sektion
# ]

# Optional: Parameter überschreiben
OVERRIDES = None
# OVERRIDES = {
#     'scenario': {'horizon': {'end': '2023-01-31 23:00'}},
#     'costs': {'co2_price_eur_per_t': 150.0}
# }

# ============================================================
# KONFIGURATION PRÜFEN
# ============================================================

print("Konfiguration:")
for cfg in CONFIG_PATHS:
    exists = (PROJECT_ROOT / cfg).exists()
    print(f"  {'OK' if exists else 'FEHLT'} {cfg}")

# Vorschau laden
cfg = load_and_merge(CONFIG_PATHS)
print(f"\nEinstellungen:")
print(f"  Workflow:   {cfg.get('scenario', {}).get('workflow', 'N/A')}")
print(f"  Solver:     {cfg.get('run', {}).get('solver', 'N/A')}")
print(f"  CO2-Preis:  {cfg.get('costs', {}).get('co2_price_eur_per_t', 'N/A')} EUR/t")
print(f"  Netzwerk:   {'Ja' if cfg.get('thermal_network', {}).get('enabled', False) else 'Nein'}")

# Optimization-Einstellungen anzeigen
opt_cfg = cfg.get('scenario', {}).get('optimization', {})
if opt_cfg:
    print(f"\nOptimization:")
    variables = opt_cfg.get('variables', {})
    hp_vars = variables.get('heat_pumps', {})
    sto_vars = variables.get('storage', {})
    print(f"  HP Capacity: {'Variable' if hp_vars.get('capacity', True) else 'Fixed'}")
    print(f"  HP Build:    {'Variable' if hp_vars.get('build', True) else 'Fixed'}")
    print(f"  Storage:     {'Variable' if sto_vars.get('capacity', True) else 'Fixed'}")
    fix_after = opt_cfg.get('fix_after_window')
    if fix_after is not None:
        print(f"  Fix After:   Window {fix_after}")

# RH-Parameter anzeigen
rh_cfg = cfg.get('scenario', {}).get('rolling_horizon', {})
if rh_cfg:
    print(f"\nRolling Horizon:")
    print(f"  Window:     {rh_cfg.get('heat_horizon_hours', 168)}h")
    print(f"  Step:       {rh_cfg.get('step_hours', 24)}h")
    print(f"  Terminal:   {rh_cfg.get('terminal_policy', 'nicht gesetzt')}")

## 3. Optimierung ausführen

In [12]:
%%time
print("="*70)
print("🚀 STARTE OPTIMIERUNG")
print("="*70)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

try:
    workflow = rh.run_workflow(CONFIG_PATHS, overrides=OVERRIDES)
    
    print("\n" + "="*70)
    print("✅ OPTIMIERUNG ERFOLGREICH")
    print("="*70)
    print(f"\n📊 Workflow: {' → '.join(workflow.plan.steps)}")
    
    optimization_success = True
    
except Exception as e:
    print("\n" + "="*70)
    print("❌ FEHLER")
    print("="*70)
    print(f"\nFehler: {e}\n")
    
    import traceback
    traceback.print_exc()
    
    workflow = None
    optimization_success = False


🚀 STARTE OPTIMIERUNG
Start: 2025-12-28 16:07:17



Design fixation requested but no PF design data available – proceeding without fixation.


[LOAD] Import_Data.xlsx → 8760 Schritte von 2023-01-01 00:00:00 bis 2023-12-31 23:00:00
[SCENARIO] Zeitraum 2023-01-01 00:00:00 → 2023-04-17 23:00:00 (2568 Schritte)
[RH] _set_initial_soc: Setting initial SOC for next window
  - soc0_mwh: 500.0 MWh
[BUILD] Storage terminal configuration:
  - terminal_state: cyclic
  - terminal_policy: equal
  - soc_init: 500.0
  - terminal_target_val: 500.0
[BUILD] Using simple storage (single-zone model)
[BUILD] Created terminal constraint: TES_terminal
  - SOC[168] == 500.0 MWh (policy: equal)
[BUILD] #el_in=5, #el_out=3, #ht_out=12, #ht_in=1
[BUILD] Thermal network disabled
[RH] _next_soc: Extracting SOC at index 167 (commit_len=168)
  - SOC value: 500.0 MWh
  - SOC range in window: [0.0, 1085.3] MWh
[RH] _set_initial_soc: Setting initial SOC for next window
  - soc0_mwh: 500.0 MWh
[BUILD] Storage terminal configuration:
  - terminal_state: cyclic
  - terminal_policy: equal
  - soc_init: 500.0
  - terminal_target_val: 500.0
[BUILD] Using simple stor

Konnte Pyomo-Wert P_buy_MW[1] nicht auslesen: No value for uninitialized VarData object P_buy[1]


ERROR: evaluating object as numeric value: P_buy[2]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object P_buy[2]


...

Konnte Pyomo-Wert P2H_Pel_MW[167] nicht auslesen: No value for uninitialized VarData object P2H_Pel[167]


ERROR: evaluating object as numeric value: P2H_Pel[168]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object P2H_Pel[168]


Konnte Pyomo-Wert P2H_Pel_MW[168] nicht auslesen: No value for uninitialized VarData object P2H_Pel[168]


ERROR: evaluating object as numeric value: HKW_fuel[1]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object HKW_fuel[1]
ERROR: evaluating object as numeric value: obj
        (object: <class 'pyomo.core.base.objective.ScalarObjective'>)
    No value for uninitialized VarData object HKW_fuel[1]

❌ FEHLER

Fehler: No value for uninitialized VarData object HKW_fuel[1]

CPU times: total: 13.1 s
Wall time: 10.7 s


Traceback (most recent call last):
  File "<timed exec>", line 7, in <module>
  File "c:\Users\LKR\Documents\GitHub\Energy_Framwork\Planing-Framework-for-Heat\calion\run\rolling_horizon.py", line 1616, in run_workflow
    handler(context)
  File "c:\Users\LKR\Documents\GitHub\Energy_Framwork\Planing-Framework-for-Heat\calion\run\rolling_horizon.py", line 1665, in _rh_step
    context.rh_result = _run_rolling_horizon(
                        ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\LKR\Documents\GitHub\Energy_Framwork\Planing-Framework-for-Heat\calion\run\rolling_horizon.py", line 1836, in _run_rolling_horizon
    window_result = _solve_scenario(window_table, window_cfg, dt_h, solver_name)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\LKR\Documents\GitHub\Energy_Framwork\Planing-Framework-for-Heat\calion\run\rolling_horizon.py", line 2056, in _solve_scenario
    series, summary, costs = _collect_timeseries_and_summary(
                  

In [ ]:
# === DEBUG: Warum ist das Modell infeasible? ===

if not optimization_success and workflow is None:
    print("\n" + "="*70)
    print("🔍 DEBUG: Infeasibility-Analyse")
    print("="*70)
    
    try:
        # 1. Config laden und prüfen
        from calion.config.merge import load_and_merge, deep_merge
        config = load_and_merge(CONFIG_PATHS)
        
        # Overrides manuell anwenden (falls vorhanden)
        if OVERRIDES:
            config = deep_merge(config, OVERRIDES)
        
        # 2. Daten laden - richtiger Key!
        from calion.io.loader import load_input_excel
        import pandas as pd
        
        input_path = config['site'].get('input_file', 'Import_Data.xlsx')
        full_input_path = PROJECT_ROOT / input_path
        
        print(f"\n📁 Input-Datei: {input_path}")
        
        if full_input_path.exists():
            # Verfügbare Sheets anzeigen
            excel_file = pd.ExcelFile(full_input_path)
            print(f"📊 Verfügbare Sheets: {excel_file.sheet_names}")
            
            # Richtigen Sheet finden (erster Sheet oder "Wärmebedarf" falls vorhanden)
            sheet_name = None
            if 'Wärmebedarf' in excel_file.sheet_names:
                sheet_name = 'Wärmebedarf'
            elif len(excel_file.sheet_names) > 0:
                sheet_name = excel_file.sheet_names[0]
                print(f"⚠️  'Wärmebedarf' nicht gefunden, verwende: '{sheet_name}'")
            
            if sheet_name:
                # Load demand
                demand_df = pd.read_excel(full_input_path, sheet_name=sheet_name, index_col=0)
                
                # Finde Wärmebedarf-Spalte
                heat_col = None
                for col in demand_df.columns:
                    if 'wärme' in col.lower() or 'waerme' in col.lower() or 'heat' in col.lower():
                        heat_col = col
                        break
                
                if heat_col:
                    demand_mw = demand_df[heat_col]
                else:
                    # Nimm erste Spalte
                    demand_mw = demand_df.iloc[:, 0]
                    print(f"⚠️  Keine Wärmebedarf-Spalte gefunden, verwende: '{demand_df.columns[0]}'")
                
                print(f"\n📊 Wärmebedarf:")
                print(f"  Spalte:        {heat_col or demand_df.columns[0]}")
                print(f"  Peak:          {demand_mw.max():.2f} MW")
                print(f"  Average:       {demand_mw.mean():.2f} MW")
                print(f"  Total (Jahr):  {demand_mw.sum():.0f} MWh")
                print(f"  Zeitschritte:  {len(demand_mw)}")
                
                # 3. Installierte Kapazität prüfen
                print(f"\n🏭 Konfigurierte Kapazitäten:")
                
                total_capacity = 0
                
                # Heat Pumps
                if 'heat_pumps' in config.get('system', {}):
                    for hp in config['system']['heat_pumps']:
                        if hp.get('enabled', True):
                            cap = hp.get('capacity_mw', hp.get('capacity_max_mw', hp.get('capacity_init_mw', 0)))
                            total_capacity += cap
                            print(f"  Heat Pump {hp['id']:10s}: {cap:.2f} MW")
                
                # Generators
                if 'generators' in config.get('system', {}):
                    for gen_id, gen_cfg in config['system']['generators'].items():
                        if gen_cfg.get('enabled', True):
                            cap = gen_cfg.get('cap_th_mw', 0)
                            total_capacity += cap
                            print(f"  Generator {gen_id:10s}: {cap:.2f} MW")
                
                # P2H
                if 'p2h' in config.get('system', {}):
                    for p2h in config['system']['p2h']:
                        if p2h.get('enabled', True):
                            cap = p2h.get('cap_th_mw', 0)
                            total_capacity += cap
                            print(f"  P2H {p2h['id']:10s}: {cap:.2f} MW")
                
                # Grid
                grid_enabled = config.get('system', {}).get('grid', {}).get('enabled', False)
                if grid_enabled:
                    grid_cap = config['system']['grid'].get('max_buy_mw', 999)
                    print(f"  Grid (Netz):   {grid_cap:.2f} MW (enabled)")
                else:
                    print(f"  Grid (Netz):   DISABLED ❌")
                
                print(f"\n📈 Bilanz:")
                print(f"  Total Capacity:  {total_capacity:.2f} MW")
                print(f"  Peak Demand:     {demand_mw.max():.2f} MW")
                print(f"  Deficit:         {max(0, demand_mw.max() - total_capacity):.2f} MW")
                
                if demand_mw.max() > total_capacity:
                    print(f"\n❌ PROBLEM: Demand ({demand_mw.max():.2f} MW) > Capacity ({total_capacity:.2f} MW)")
                    print(f"\n💡 LÖSUNG:")
                    print(f"   1. Erhöhen Sie capacity_mw in stadtbach.yaml")
                    print(f"   2. ODER aktivieren Sie Grid: grid.enabled = true")
                    print(f"   3. ODER aktivieren Sie mehr Erzeuger")
                else:
                    print(f"\n✅ Kapazität ist ausreichend")
                    print(f"\n⚠️  Problem liegt vermutlich woanders:")
                    print(f"   - Prüfen Sie Storage-Constraints (e_min, e_max, soc0)")
                    print(f"   - Prüfen Sie COP-Werte (nicht 0 oder negativ)")
                    print(f"   - Prüfen Sie min_load Constraints")
                
                # 4. Storage prüfen
                if 'storage' in config.get('system', {}):
                    storage = config['system']['storage']
                    if storage.get('enabled', False):
                        print(f"\n🔋 Speicher-Konfiguration:")
                        e_min = storage.get('e_min', storage.get('min_energy_mwh', 0))
                        e_max = storage.get('e_max', storage.get('max_energy_mwh', storage.get('energy_mwh', 0)))
                        soc0 = storage.get('soc0', storage.get('soc0_mwh', 0))
                        print(f"  e_min:  {e_min:.2f} MWh")
                        print(f"  e_max:  {e_max:.2f} MWh")
                        print(f"  soc0:   {soc0:.2f} MWh")
                        
                        if soc0 > e_max:
                            print(f"  ❌ PROBLEM: soc0 > e_max!")
            else:
                print(f"❌ Keine Sheets in Excel-Datei gefunden!")
        else:
            print(f"❌ Input-Datei nicht gefunden: {full_input_path}")
    
    except Exception as e:
        print(f"\n❌ Debug-Fehler: {e}")
        import traceback
        traceback.print_exc()
    
    print("\n" + "="*70)

## 4. Ergebnisse speichern

In [ ]:
if optimization_success and workflow:
    workflow_dir = save_workflow_run(
        workflow,
        name="CALION Simulation",
        description="Optimierungslauf",
        config_paths=CONFIG_PATHS
    )
    print(f"\n📂 Gespeichert in: {workflow_dir}")
else:
    print("⚠️ Keine Ergebnisse zum Speichern")

## 5. Zusammenfassung & KPIs

In [ ]:
if optimization_success and workflow:
    display_workflow_summary(workflow)
    print("\n")
    display_kpi_summary(workflow)
else:
    print("⚠️ Keine Ergebnisse verfügbar")

## 6. Thermisches Netzwerk

In [ ]:
if optimization_success and workflow:
    network_enabled = workflow.cfg.get('thermal_network', {}).get('enabled', False)
    
    if network_enabled:
        print("🌡️ THERMISCHES NETZWERK")
        print("=" * 70)
        
        result = workflow.pf_result or workflow.rh_result
        
        if result and hasattr(result, 'summary') and 'thermal_network' in result.summary:
            net = result.summary['thermal_network']
            
            print(f"\n📏 Topologie:")
            print(f"  Knoten:        {net.get('Number_of_nodes', 0)}")
            print(f"  Rohre:         {net.get('Number_of_pipes', 0)}")
            print(f"  Länge:         {net.get('Total_pipe_length_m', 0):.0f} m")
            
            print(f"\n🔥 Wärme:")
            print(f"  Geliefert:     {net.get('Total_heat_delivered_MWh', 0):.1f} MWh")
            print(f"  Verluste:      {net.get('Total_heat_loss_MWh', 0):.1f} MWh")
            print(f"  Verlustrate:   {net.get('Heat_loss_percentage', 0):.2f}%")
        else:
            print("\n⚠️ Netzwerk-Ergebnisse nicht verfügbar")
            print("   (Network summary data ist nur für PF-Runs verfügbar,")
            print("    nicht für RH-Runs. Verwenden Sie workflow='PF' zum Testen.)")
    else:
        print("ℹ️ Thermisches Netzwerk nicht aktiviert")
        print("   Aktivieren in configs/stadtbach.yaml:")
        print("   thermal_network:")
        print("     enabled: true")

## 7. Visualisierung

In [ ]:
if optimization_success and workflow:
    from calion.io.publication_plotter import export_publication_plots
    
    result = workflow.pf_result or workflow.rh_result
    
    if result:
        print("📊 Erstelle Plots...")
        
        plots = export_publication_plots(
            outdir=str(workflow_dir) if 'workflow_dir' in dir() else 'exports',
            table=result.table,
            series=result.series,
            summary_sections=result.summary if hasattr(result, 'summary') else {},
            dpi=150,
            formats=("png",),
            plot_types=["heat_balance", "electric_balance", "storage"]
        )
        
        print(f"✅ {len(plots)} Plots erstellt")
else:
    print("⚠️ Keine Daten für Visualisierung")

## 8. Interaktive Zeitreihen

In [ ]:
if optimization_success and workflow:
    result = workflow.pf_result or workflow.rh_result
    
    if result:
        # DataFrame erstellen aus table und series
        ts_dict = {'timestamp': result.table.index}
        
        # Daten aus TimeSeriesTable hinzufügen
        for col in result.table.columns:
            ts_dict[col] = [result.table[col][i] for i in range(len(result.table))]
        
        # Series hinzufügen
        ts_dict.update(result.series)
        
        ts = pd.DataFrame(ts_dict)
        ts.set_index('timestamp', inplace=True)
        
        print(f"📋 Zeitreihen: {len(ts)} Zeilen, {len(ts.columns)} Spalten")
        print(f"\nVerfügbare Spalten:")
        for i, col in enumerate(ts.columns[:20]):
            print(f"  {col}")
        if len(ts.columns) > 20:
            print(f"  ... und {len(ts.columns) - 20} weitere")
        
        display(ts.head())

In [ ]:
# Einfacher Plot mit matplotlib
if optimization_success and workflow and 'ts' in dir():
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Wärmebedarf
    if 'waermebedarf_MWth' in ts.columns:
        axes[0].plot(ts.index, ts['waermebedarf_MWth'], label='Wärmebedarf', color='red', alpha=0.7)
        axes[0].set_ylabel('Leistung [MW]')
        axes[0].legend()
        axes[0].set_title('Wärmebedarf')
    elif 'heatd' in ts.columns:
        axes[0].plot(ts.index, ts['heatd'], label='Wärmebedarf', color='red', alpha=0.7)
        axes[0].set_ylabel('Leistung [MW]')
        axes[0].legend()
        axes[0].set_title('Wärmebedarf')
    
    # Strompreis
    if 'strompreis_EUR_MWh' in ts.columns:
        axes[1].plot(ts.index, ts['strompreis_EUR_MWh'], label='Strompreis', color='blue', alpha=0.7)
        axes[1].set_ylabel('Preis [EUR/MWh]')
        axes[1].legend()
        axes[1].set_title('Strompreis')
    elif 'price' in ts.columns:
        axes[1].plot(ts.index, ts['price'], label='Strompreis', color='blue', alpha=0.7)
        axes[1].set_ylabel('Preis [EUR/MWh]')
        axes[1].legend()
        axes[1].set_title('Strompreis')
    
    plt.tight_layout()
    plt.show()

## 9. Dashboard starten

Für detaillierte interaktive Analyse:

In [ ]:
print("🎛️ Dashboard starten:")
print("\n  python start_dashboard.py")
print("\nOder direkt hier (nur in Jupyter):")

🎛️ Dashboard starten:

  python start_dashboard.py

Oder direkt hier (nur in Jupyter):


In [ ]:
# Dashboard im Notebook anzeigen (optional)
# Auskommentieren um zu aktivieren:

# if optimization_success and workflow:
#     from calion.io.dashboard import create_dashboard
#     dashboard = create_dashboard(workflow)
#     dashboard.servable()

## 10. Gespeicherte Workflows laden

In [ ]:
# Liste aller gespeicherten Workflows
saved = list_saved_workflows(sort_by="date")

print(f"📦 {len(saved)} gespeicherte Workflows:\n")
for i, wf in enumerate(saved[:5], 1):
    name = wf['name'][:50]
    print(f"  {i}. {name}")

📦 7 gespeicherte Workflows:

  1. CALION Simulation
  2. CALION Simulation
  3. CALION Simulation
  4. CALION Simulation
  5. CALION Simulation


In [ ]:
# Workflow laden (Index anpassen)
# LOAD_INDEX = 0  # Erster Workflow
# 
# if saved:
#     loaded_workflow = load_workflow_from_saved(saved[LOAD_INDEX]['path'])
#     print(f"✅ Geladen: {saved[LOAD_INDEX]['name']}")
#     display_workflow_summary(loaded_workflow)

---

## Hilfe

**Dokumentation:**
- `README.md` - Übersicht
- `docs/methodology.md` - Methodik

**CLI-Nutzung:**
```bash
python -m calion.run configs/stadtbach.yaml
```

**Dashboard als Webapp:**
```bash
python start_dashboard.py
```